In [2]:
import pandas as pd
import math
import re

# PARAMETROS METODOLOGICOS

In [3]:
# En base al programa minimo DS 49, se hizo una suma de recintos(lugares de una casa), y se calculo el porcentaje
# del factor dormitorio que dio 0,44 (Cuadro Normativo y Tabla de Espacios y Usos Mínimos)
FACTOR_DORMITORIOS = {
    "departamento": 0.50,  # plantas más eficientes, algunos recintos más compactos -> cocina y bañp
    "casa":         0.40,  # pasillos mas largos, bodega, estacionamiento, etc
}
M2_MINIMO_DORMITORIO_MINVU = 8  # m² mínimo por dormitorio (MINVU)

# CENSO INE: personas promedio por hogar por comuna 
PERSONAS_POR_HOGAR = {
    "Valdivia":    2.9,
    "Lanco":       3.2,
    "Los Lagos":   3.2,
    "Máfil":       3.1,
    "Corral":      3.0,
    "Futrono":     3.2,
    "La Unión":    3.1,
    "Río Bueno":   3.2,
    "Panguipulli": 3.2,
    "Paillaco":    3.1,
    "Mariquina":   3.1,
}

# FUNCIONES PARA CLASIFICAR

In [4]:
def clasificar_hacinamiento(indice: float) -> tuple:
    if indice < 2.5:
        return 1, "Sin hacinamiento"
    elif indice < 3.5:
        return 2, "Hacinamiento medio"
    elif indice < 5.0:
        return 3, "Hacinamiento crítico"
    else:
        return 4, "Hacinamiento severo"


def calcular_hacinamiento(m2_utiles: float, tipo: str, comuna: str) -> dict:
    """
    Calcula el índice de hacinamiento f(A) para una propiedad.
    """
    tipo   = tipo.lower().strip()
    comuna = comuna.strip()

    # Personas por hogar (CENSO )
    personas = PERSONAS_POR_HOGAR.get(comuna, 3.1)

    # Dormitorios estimados (mínimo 1)
    factor         = FACTOR_DORMITORIOS.get(tipo, 0.45)
    m2_dormitorios = m2_utiles * factor
    dormitorios    = max(1, math.floor(m2_dormitorios / M2_MINIMO_DORMITORIO_MINVU))

    # Índice A y clasificación f(A)
    indice              = round(personas / dormitorios, 2)
    f_a, clasificacion  = clasificar_hacinamiento(indice)

    return {
        "dormitorios_estimados": dormitorios,
        "personas_hogar":        personas,
        "indice_hacinamiento":   indice,
        "f_A":                   f_a,
        "clasificacion":         clasificacion,
    }

# LIMPIAR DATOS CSV SCRAPP

In [5]:
def limpiar_precio(precio_str: str) -> float:
    try:
        # Eliminar puntos de miles, espacios y símbolo $
        limpio = re.sub(r"[.\s]", "", str(precio_str)).replace(",", "").replace("$", "")
        return float(limpio)
    except (ValueError, AttributeError):
        return None


def limpiar_m2(m2_str: str) -> float:
    try:
        numeros = re.findall(r"\d+", str(m2_str))
        return float(numeros[0]) if numeros else None
    except (ValueError, AttributeError, IndexError):
        return None

# Cargar CSV y aplicar limpieza

In [6]:
RUTA_CSV = "propiedades_los_rios.csv"

df = pd.read_csv(RUTA_CSV)
df.head()

,id,tipo,modalidad,comuna,precio,m2_utiles,ubicacion
0,1,casa,arriendo,Valdivia,1.100.000,200 m² útiles,"Hernando De Rivera 300 - 600, Valdivia, Sur De..."
1,2,casa,arriendo,Valdivia,640.000,73 m² útiles,"6W62+5H, Valdivia, Valdivia"
2,3,casa,arriendo,Valdivia,3000000,300 m² útiles,"Carampangue, Valdivia, Centro De Valdivia, Val..."
3,4,casa,arriendo,Valdivia,1.350.000,172 m² útiles,"Nebuco 1 - 300, Valdivia, Sur De Valdivia, Val..."
4,5,casa,arriendo,Valdivia,2000000,170 m² útiles,"O'Higgins 234, Valdivia, Centro De Valdivia, V..."


In [7]:
# Limpiar directamente sobre las columnas originales
df["precio"]    = df["precio"].apply(limpiar_precio)
df["m2_utiles"] = df["m2_utiles"].apply(limpiar_m2)

# Registros con m² inválidos
n_invalidos = df["m2_utiles"].isna().sum()

# Filtrar registros válidos
df = df.dropna(subset=["m2_utiles"]).copy()
df = df[df["m2_utiles"] > 0].copy()
df = df.reset_index(drop=True)

df.head()

,id,tipo,modalidad,comuna,precio,m2_utiles,ubicacion
0,1,casa,arriendo,Valdivia,1100000.0,200.0,"Hernando De Rivera 300 - 600, Valdivia, Sur De..."
1,2,casa,arriendo,Valdivia,640000.0,73.0,"6W62+5H, Valdivia, Valdivia"
2,3,casa,arriendo,Valdivia,3000000.0,300.0,"Carampangue, Valdivia, Centro De Valdivia, Val..."
3,4,casa,arriendo,Valdivia,1350000.0,172.0,"Nebuco 1 - 300, Valdivia, Sur De Valdivia, Val..."
4,5,casa,arriendo,Valdivia,2000000.0,170.0,"O'Higgins 234, Valdivia, Centro De Valdivia, V..."


# Calcular hacinamiento por propiedad

In [8]:
# Aplicar la función fila por fila
resultados = df.apply(
    lambda row: calcular_hacinamiento(
        m2_utiles = row["m2_utiles"],
        tipo      = row["tipo"],
        comuna    = row["comuna"],
    ),
    axis=1,
    result_type="expand",
)

# Unir resultados al DataFrame
df = df.join(resultados)

# Calcular precio por m²
df["precio_m2"] = df["precio"] / df["m2_utiles"]

print(f"Columnas del DataFrame final: {df.columns.tolist()}")
df[["tipo", "comuna", "m2_utiles", "precio", "precio_m2",
    "dormitorios_estimados", "personas_hogar",
    "indice_hacinamiento", "f_A", "clasificacion"]].head(10)

Columnas del DataFrame final: ['id', 'tipo', 'modalidad', 'comuna', 'precio', 'm2_utiles', 'ubicacion', 'dormitorios_estimados', 'personas_hogar', 'indice_hacinamiento', 'f_A', 'clasificacion', 'precio_m2']


,tipo,comuna,m2_utiles,precio,precio_m2,dormitorios_estimados,personas_hogar,indice_hacinamiento,f_A,clasificacion
0,casa,Valdivia,200.0,1100000.0,5500.000000,10,2.9,0.29,1,Sin hacinamiento
1,casa,Valdivia,73.0,640000.0,8767.123288,3,2.9,0.97,1,Sin hacinamiento
2,casa,Valdivia,300.0,3000000.0,10000.000000,15,2.9,0.19,1,Sin hacinamiento
3,casa,Valdivia,172.0,1350000.0,7848.837209,8,2.9,0.36,1,Sin hacinamiento
4,casa,Valdivia,170.0,2000000.0,11764.705882,8,2.9,0.36,1,Sin hacinamiento
5,casa,Valdivia,114.0,850000.0,7456.140351,5,2.9,0.58,1,Sin hacinamiento
6,casa,Valdivia,240.0,1750000.0,7291.666667,12,2.9,0.24,1,Sin hacinamiento
7,casa,Valdivia,65.0,350000.0,5384.615385,3,2.9,0.97,1,Sin hacinamiento
8,casa,Valdivia,260.0,1400000.0,5384.615385,13,2.9,0.22,1,Sin hacinamiento
9,casa,Valdivia,220.0,800000.0,3636.363636,11,2.9,0.26,1,Sin hacinamiento


In [9]:
dist = (
    df.groupby(["f_A", "clasificacion"])
    .size()
    .reset_index(name="n")
)
dist["pct"] = (dist["n"] / len(df) * 100).round(1)

print("=== Distribución f(A) — Región de Los Ríos ===")
print(dist.to_string(index=False))
print(f"\nÍndice A promedio región: {df['indice_hacinamiento'].mean():.2f}")

=== Distribución f(A) — Región de Los Ríos ===
 f_A      clasificacion    n  pct
   1   Sin hacinamiento 1539 94.9
   2 Hacinamiento medio   83  5.1

Índice A promedio región: 0.68


In [10]:
SALIDA_PROPIEDADES = "propiedades_con_hacinamiento.csv"

df.to_csv(SALIDA_PROPIEDADES, index=False)
print(f"✓ Exportado: {SALIDA_PROPIEDADES}  ({len(df)} filas)")

✓ Exportado: propiedades_con_hacinamiento.csv  (1622 filas)


In [13]:
resumen = (
    df.groupby(["comuna", "modalidad"])
    .agg(
        n_propiedades     = ("id", "count"),
        m2_promedio       = ("m2_utiles", "mean"),
        dormitorios_prom  = ("dormitorios_estimados", "mean"),
        indice_promedio   = ("indice_hacinamiento", "mean"),
        f_A_promedio      = ("f_A", "mean"),
        pct_sin_hacin     = ("clasificacion", lambda x: (x == "Sin hacinamiento").mean()    * 100),
        pct_hacin_medio   = ("clasificacion", lambda x: (x == "Hacinamiento medio").mean()  * 100),
        pct_hacin_critico = ("clasificacion", lambda x: (x == "Hacinamiento crítico").mean() * 100),
        pct_hacin_severo  = ("clasificacion", lambda x: (x == "Hacinamiento severo").mean()  * 100),
        precio_promedio   = ("precio", "mean"),
        precio_m2_mediana = ("precio_m2", "median"),
    )
    .round(2)
    .reset_index()
)
resumen

,comuna,modalidad,n_propiedades,m2_promedio,dormitorios_prom,indice_promedio,f_A_promedio,pct_sin_hacin,pct_hacin_medio,pct_hacin_critico,pct_hacin_severo,precio_promedio,precio_m2_mediana
0,Corral,arriendo,1,72.00,3.00,1.00,1.00,100.00,0.00,0.0,0.0,4.000000e+05,5555.56
1,Corral,venta,2,130.00,6.50,0.65,1.00,100.00,0.00,0.0,0.0,4.175000e+08,2758333.33
2,Futrono,arriendo,18,206.72,10.22,0.40,1.00,100.00,0.00,0.0,0.0,5.988889e+05,2475.00
3,Futrono,venta,91,244.30,12.01,0.45,1.03,96.70,3.30,0.0,0.0,5.880475e+08,2181818.18
4,La Unión,arriendo,3,63.33,3.00,1.12,1.00,100.00,0.00,0.0,0.0,4.333333e+05,6363.64
5,La Unión,venta,54,189.31,9.11,0.64,1.06,94.44,5.56,0.0,0.0,3.065259e+08,1230041.90
6,Lanco,arriendo,1,140.00,7.00,0.46,1.00,100.00,0.00,0.0,0.0,4.000000e+05,2857.14
7,Lanco,venta,9,232.44,11.44,0.42,1.00,100.00,0.00,0.0,0.0,3.199822e+08,1129411.76
8,Los Lagos,arriendo,2,90.00,4.50,0.72,1.00,100.00,0.00,0.0,0.0,4.500000e+05,4937.50
9,Los Lagos,venta,39,164.90,7.92,0.67,1.03,97.44,2.56,0.0,0.0,2.262667e+08,1084337.35


In [14]:
SALIDA_RESUMEN = "resumen_hacinamiento_comunas.csv"

resumen.to_csv(SALIDA_RESUMEN, index=False)
print(f"  Propiedades procesadas : {len(df)}")


✓ Exportado: resumen_hacinamiento_comunas.csv  (21 filas)

=== Resumen completado ===
  Propiedades procesadas : 1622
  Comunas                : 11
  Modalidades            : <ArrowStringArray>
['arriendo', 'venta']
Length: 2, dtype: str
